# Resampling‑related processing and visualization

You can execute the example code at the end of the notebook to reproduce
the behaviour of the `main()` function defined in the original script.

In [1]:
%pwd

'c:\\Users\\aurel\\Projects\\deezer-robustness-project\\robust-deepfake-detector\\scripts\\attack\\resampling'

In [ ]:
# standard imports
import sys
sys.argv = ["notebook"]
from pathlib import Path

# add project root to path so we can import deezer.compute_fakeprints
# in notebooks __file__ may be undefined; use current working dir
project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import numpy as np
import torchaudio
import torch

from references.deezer.compute_fakeprints import open_audio_slice, fakeprint
import soxr

# enable inline plotting for notebooks
%matplotlib inline


usage: ipykernel_launcher.py [-h] [--save SAVE] [--path PATH] [--sr SR]
                             [--fmin FMIN] [--fmax FMAX]
ipykernel_launcher.py: error: ambiguous option: --f=c:\Users\aurel\AppData\Roaming\Code\User\globalStorage\ms-toolsai.jupyter\version-2025.7.0\jupyter\runtime\kernel-v30ca37dcddc2a6ccafc8d54aa638c0e6b84370fbf.json could match --fmin, --fmax


SystemExit: 2

## Constants and parameters

The original script defined a handful of global parameters; we keep them
here so you can change them interactively and re‑execute the cells.

In [ ]:
FREQ = 1000       # frequency of test sinusoid (Hz)
DURATION = 10.0   # duration in seconds
SR = 44100        # original sampling rate (Hz)
FMIN = 5000       # fakeprint lower bound (Hz)
FMAX = 16000      # fakeprint upper bound (Hz)
N_FFT = 4096      # FFT window size (power of two)


## `get_fft`

Load an audio file (MP3, WAV, …) using `torchaudio`.  If the file has
multiple channels it is averaged to mono.  The sample rate is optionally
changed using `soxr`.  Finally the real‑valued FFT magnitude spectrum and
corresponding frequency axis are returned.

In [ ]:
def get_fft(mp3_path, sr):
    """
    Return (freqs, fft_mag) for the audio at ``mp3_path`` resampled to
    ``sr`` Hz.
    """
    audio, orig_sr = torchaudio.load(mp3_path, channels_first=True, normalize=True)
    
    # mix to mono if necessary
    if audio.shape[0] > 1:
        audio = torch.mean(audio, dim=0, keepdim=True)

    if orig_sr != sr:
        # convert to numpy, resample with soxr, convert back to torch
        audio_np = audio.numpy()
        audio_np = soxr.resample(audio_np.T, orig_sr, sr).T
        audio = torch.from_numpy(audio_np)
    
    audio = audio.flatten()
    n = len(audio)

    fft_complex = torch.fft.rfft(audio)
    fft_mag = torch.abs(fft_complex).numpy()
    freqs = torch.fft.rfftfreq(n, 1/sr).numpy()
    
    return freqs, fft_mag


## `spectrogram`

Load an audio slice using `deezer.compute_fakeprints.open_audio_slice`,
optionally resample and clip long files, compute a power spectrogram
using `torchaudio.transforms.Spectrogram`, and convert the result to
decibels.

In [ ]:
def spectrogram(f_name, max_duration=180, SR=44100, n_fft=4096):
    p, sr = open_audio_slice(f_name)
    
    # resample to requested rate
    if sr != SR:
        p = soxr.resample(p, sr, SR)
    
    # truncate to maximum duration (in seconds)
    p = p[:SR * max_duration]

    # create transformer with the current FFT size
    transformer = torchaudio.transforms.Spectrogram(n_fft=n_fft, power=2)
    stft = transformer(torch.Tensor(p.T)).numpy()
    
    # convert to dB, clip to avoid log(0)
    stft_db = 10 * np.log10(np.clip(stft, 1e-10, 1e6))
    
    return stft_db


## `visualize_fft`

Plot the FFT magnitude in decibels with properly labelled axes.  The
frequency range is limited to `[0, sr/2]` and a small margin is added for
readability.

In [ ]:
def visualize_fft(freqs, fft_mag, sr):
    fmin = 0
    fmax = int(sr/2)
    fft_db = 20 * np.log10(fft_mag + 1e-10)

    plt.figure(figsize=(10, 6))
    plt.plot(freqs, fft_db)
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Magnitude (dB)')
    plt.title(f'FFT Spectrum (SR={sr} Hz)')
    plt.grid(True, alpha=0.3)
    plt.xlim(fmin - 1000, fmax + 1000)
    plt.tight_layout()
    plt.show()


## `visualize_spectrogram`

Show a spectrogram array with correct time and frequency axes and
optionally plot the mean frequency profile (averaged over time).

In [ ]:
def visualize_spectrogram(spectro, sr, n_fft, title="Spectrogram", viz_mean=False):
    # if there are channels, take the first
    if spectro.ndim == 3:
        spectro_shaped = spectro[0]
    else:
        spectro_shaped = spectro
    
    hop_length = n_fft // 2
    num_frames = spectro_shaped.shape[1]
    duration = (num_frames * hop_length) / sr
    nyquist = sr / 2

    plt.figure(figsize=(12, 6))
    plt.imshow(spectro_shaped, origin='lower', aspect='auto',
               extent=[0, duration, 0, nyquist], cmap='inferno')
    plt.colorbar(format='%+2.0f dB')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Frequency (Hz)')
    plt.title(title)
    plt.tight_layout()
    plt.show()

    if viz_mean:
        stft_freqs = torch.fft.rfftfreq(N_FFT, 1/SR).numpy()
        mean_stft = np.mean(spectro, axis=2)
        plt.figure(figsize=(10, 5))
        plt.plot(stft_freqs, mean_stft[0, :], label="Mean STFT")
        plt.xlabel('Frequency (Hz)')
        plt.ylabel('Magnitude (dB)')
        plt.title(f'Averaged Spectrogram (N_FFT={N_FFT})')
        plt.grid(True, alpha=0.3)
        plt.show()


## `visualize_fp`

Plot a fakeprint curve over a given frequency axis.

In [ ]:
def visualize_fp(x_axis, fp, sr):
    plt.figure(figsize=(10, 5))
    plt.plot(x_axis, fp)
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Fakeprint Amplitude (Normalized)')
    plt.title(f'Fakeprint ({x_axis.min():.0f}-{x_axis.max():.0f} Hz)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## Example usage

Reproduce the behaviour of `main()` from the original script.  You can
modify `mp3_path` or any of the parameters above and re‑run.

In [ ]:
mp3_path = "data/signals/signal2_spikes_noise.wav"

# compute FFT and plot
freqs, fft_mag = get_fft(mp3_path, SR)
visualize_fft(freqs, fft_mag, SR)

# compute and display spectrogram
spectro = spectrogram(mp3_path, SR=SR, n_fft=N_FFT)
visualize_spectrogram(spectro, SR, N_FFT, viz_mean=True)

# compute fakeprint and visualise it on the correct frequency axis
fp = fakeprint(spectro, f_range=[FMIN, FMAX], SR=SR)
num_bins = spectro.shape[1]
full_freqs = np.linspace(0, SR / 2, num=num_bins)
mask = (FMIN < full_freqs) & (full_freqs < FMAX)
fp_freqs = full_freqs[mask]
if len(fp_freqs) != len(fp):
    print(f"Warning: Size mismatch: axis {len(fp_freqs)}, data {len(fp)}")
    visualize_fp(np.linspace(FMIN, FMAX, len(fp)), fp, SR)
else:
    visualize_fp(fp_freqs, fp, SR)
